# Convergence du calcul axisymétrique AMR

Ce notebook lit uniquement `campaign/`. Lancer les calculs avec `run_campaign.sh`, puis exécuter les cellules dans l'ordre. Les cases absentes, interrompues, échouées ou invalides restent sans erreur calculée.

La référence **provisoire** utilise `MAXLEVEL=12`, `MINLEVEL=5`, `F_TOL=1e-3`, `U_TOL=1e-5`. Elle doit encore être vérifiée : la résolution maximale et la tolérance fixe sur l'interface peuvent limiter sa précision.

À $t=0{,}15$ s, on compare la composante axiale $u_x$ aux mêmes 1024 × 64 points intérieurs du tube. L'erreur maximale est une estimation **sur cette grille d'échantillonnage**, et non le supremum continu : des extrema plus fins peuvent être manqués. Coordonnées en mètres, vitesses en m/s, hauteurs en mètres.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

campaign = Path('campaign')  # ouvrir Jupyter depuis la racine du projet
levels = np.arange(6, 13)
tolerance_labels = ['1e-2', '3e-3', '1e-3', '3e-4', '1e-4']
tolerances = np.array([float(value) for value in tolerance_labels])

On accepte uniquement les dossiers marqués `complete`, avec des paramètres et des sorties cohérents. Les coordonnées sont comparées exactement, dans l'ordre `x` axial puis `y` radial. Une valeur absente, non finie ou égale au marqueur Basilisk `nodata` invalide le résultat ; elle n'est jamais remplacée par zéro.

In [ ]:
def read_run(directory, level, tolerance):
    result = {'status': 'absent', 'directory': directory, 'level': level, 'tolerance': tolerance}
    if not directory.exists():
        return result
    status_file = directory / 'status'
    result['status'] = status_file.read_text().strip() if status_file.exists() else 'non vérifié'
    if result['status'] != 'complete':
        return result
    try:
        p = json.loads((directory / 'parametres.json').read_text())
        expected = dict(minlevel=5, maxlevel=int(level), f_tol=1e-3, u_tol=float(tolerance),
                        t_end=0.8, t_compare=0.15, length_m=0.08, radius_m=0.005,
                        sample_nx=1024, sample_ny=64)
        if p != expected:
            raise ValueError('paramètres différents de la campagne')
        source_hash = (directory / 'source.sha256').read_text().strip()
        if len(source_hash) != 64 or any(c not in '0123456789abcdef' for c in source_hash):
            raise ValueError('empreinte de source absente ou invalide')
        h = np.loadtxt(directory / 'hauteur_axi_amr.dat', ndmin=2)
        c = np.loadtxt(directory / 'cellules.dat', ndmin=2)
        v = np.loadtxt(directory / 'vitesse_t015.dat', ndmin=2)
        if any(not np.isfinite(a).all() or np.any(np.abs(a) >= 1e29) for a in (h, c, v)):
            raise ValueError('valeur non finie ou nodata')
        if h.shape != (1601, 3) or not np.allclose(h[:, 0], np.arange(1601)*0.0005, rtol=0, atol=1e-10):
            raise ValueError('série de hauteur incomplète')
        if np.any(h[:, 1:] < 0) or np.any(h[:, 1:] > p['length_m']):
            raise ValueError('hauteur hors du tube')
        if c.shape != (81, 2) or not np.allclose(c[:, 0], np.arange(81)*0.01, rtol=0, atol=1e-10):
            raise ValueError('série de cellules incomplète')
        if (c[0, 1] != 64 or np.any(c[:, 1] < 64) or np.any(c[:, 1] != np.floor(c[:, 1]))
                or np.any(c[:, 1] > 2**(2*int(level) - 4))):
            raise ValueError('comptage des cellules incohérent')
        nx, ny = p['sample_nx'], p['sample_ny']
        xx = np.repeat((np.arange(nx) + 0.5)*p['length_m']/nx, ny)
        yy = np.tile((np.arange(ny) + 0.5)*p['radius_m']/ny, nx)
        if v.shape != (nx*ny, 3) or not np.array_equal(v[:, :2], np.column_stack((xx, yy))):
            raise ValueError('grille différente de la grille physique commune')
        with (directory / 'vitesse_t015.dat').open() as stream:
            sample_time = float(stream.readline().split('=', 1)[1])
        if abs(sample_time - p['t_compare']) > 1e-12:
            raise ValueError('instant de comparaison incorrect')
        timing = dict(line.split() for line in (directory / 'temps.dat').read_text().splitlines())
        wall = float(timing['real_s'])
        if not np.isfinite(wall) or wall < 0:
            raise ValueError('temps réel invalide')
        # Moyenne temporelle par la règle des trapèzes sur [0, 0.8] s.
        mean_cells = np.sum(np.diff(c[:, 0])*(c[:-1, 1] + c[1:, 1])/2)/(c[-1, 0] - c[0, 0])
        result.update(source_hash=source_hash, parameters=p, height=h, cells=c, velocity=v, mean_cells=mean_cells,
                      wall=wall, max_rss_kb=float(timing.get('max_rss_kb', 'nan')))
    except (OSError, ValueError, KeyError, IndexError) as error:
        result['status'] = f'invalide : {error}'
    return result


def compare_runs(run, reference):
    if not np.array_equal(run['velocity'][:, :2], reference['velocity'][:, :2]):
        raise ValueError('Les deux grilles de vitesse ne sont pas identiques')
    velocity_error = np.max(np.abs(run['velocity'][:, 2] - reference['velocity'][:, 2]))
    h, hr = run['height'], reference['height']
    if not (np.isclose(h[0, 0], hr[0, 0]) and np.isclose(h[-1, 0], hr[-1, 0])):
        raise ValueError('Les séries temporelles ne couvrent pas le même intervalle')
    # L'union des instants donne le maximum des deux interpolants linéaires.
    times = np.union1d(h[:, 0], hr[:, 0])
    height_error = np.max(np.abs(np.interp(times, h[:, 0], h[:, 1])
                                 - np.interp(times, hr[:, 0], hr[:, 1])))
    return float(velocity_error), float(height_error)


reference = read_run(campaign / 'reference', 12, 1e-5)
runs = {}
for level in levels:
    for label, tolerance in zip(tolerance_labels, tolerances):
        name = f'L{level:02d}_U{label}'
        run = read_run(campaign / name, level, tolerance)
        if run['status'] == 'complete' and reference['status'] == 'complete':
            if run['source_hash'] != reference['source_hash']:
                run['status'] = 'incompatible : source différente de la référence'
            else:
                run['error_u'], run['error_h'] = compare_runs(run, reference)
        runs[(int(level), label)] = run

print('Référence :', reference['status'])
print('Configurations valides :', sum(r['status'] == 'complete' for r in runs.values()), '/ 35')

## Carte d'erreur sur la vitesse

La couleur représente $\log_{10}[E_\infty/(1\,\mathrm{m/s})]$. Chaque case donne cette valeur et le nombre moyen de cellules actives entre parenthèses. Une erreur exactement nulle est annotée $-\infty$ et utilise la couleur minimale, sans lui attribuer une erreur artificielle. Le tiret indique une erreur indisponible, notamment tant que la référence est absente.

In [ ]:
errors = np.full((len(levels), len(tolerances)), np.nan)
mean_cells = np.full_like(errors, np.nan)
for i, level in enumerate(levels):
    for j, label in enumerate(tolerance_labels):
        run = runs[(int(level), label)]
        if run['status'] == 'complete':
            mean_cells[i, j] = run['mean_cells']
            errors[i, j] = run.get('error_u', np.nan)

log_errors = np.full_like(errors, np.nan)
positive = np.isfinite(errors) & (errors > 0)
log_errors[positive] = np.log10(errors[positive])
log_errors[errors == 0] = -np.inf
finite_logs = log_errors[np.isfinite(log_errors)]
vmin, vmax = (float(finite_logs.min()), float(finite_logs.max())) if finite_logs.size else (-1., 0.)
if vmin == vmax:
    vmin, vmax = vmin - 0.5, vmax + 0.5
color_values = np.where(np.isneginf(log_errors), vmin, log_errors)
x = 1/tolerances
xedges = np.r_[x[0]/np.sqrt(x[1]/x[0]), np.sqrt(x[:-1]*x[1:]), x[-1]*np.sqrt(x[-1]/x[-2])]
fig, ax = plt.subplots(figsize=(10, 6), layout='constrained')
cmap = plt.colormaps['viridis'].copy()
cmap.set_bad('#eeeeee')
image = ax.pcolormesh(xedges, np.arange(5.5, 13), np.ma.masked_invalid(color_values),
                      cmap=cmap, vmin=vmin, vmax=vmax, edgecolors='white', linewidth=1.)
for i, level in enumerate(levels):
    for j, xpos in enumerate(x):
        value = log_errors[i, j]
        label = '−∞' if np.isneginf(value) else f'{value:.2f}' if np.isfinite(value) else '—'
        if np.isfinite(mean_cells[i, j]):
            label += f'\n({mean_cells[i, j]:,.0f})'.replace(',', ' ')
        color = 'white' if not np.isnan(value) and color_values[i, j] < (vmin + vmax)/2 else 'black'
        ax.text(xpos, level, label, ha='center', va='center', color=color, fontsize=9)
ax.set(xscale='log', yticks=levels, ylabel='MAXLEVEL', xlabel=r'$1/U_{\mathrm{TOL}}$ (s/m)',
       title='Erreur maximale sur la vitesse axiale à t = 0,15 s')
ax.set_xticks(x, [f'{value:g}' for value in x])
bar = fig.colorbar(image, ax=ax, label=r'$\log_{10}[E_\infty/(1\,\mathrm{m/s})]$')
if not np.isfinite(errors).any():
    bar.set_ticks([])
    bar.set_label('Erreurs indisponibles')
elif not finite_logs.size:
    bar.set_ticks([vmin], ['−∞'])
plt.show()

## Coût numérique

L'erreur est comparée au nombre moyen de cellules physiques actives. Les temps réels figurent dans le tableau final ; le pic de RAM, s'il est disponible, est conservé dans `temps.dat` (kio). Les points à erreur strictement positive sont représentés sur les axes logarithmiques.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), layout='constrained')
for level in levels:
    selected = [run for (lev, _), run in runs.items()
                if lev == level and run.get('error_u', np.nan) > 0]
    if selected:
        ax.scatter([r['mean_cells'] for r in selected], [r['error_u'] for r in selected], label=f'L{level}')
ax.set(xscale='log', yscale='log', xlabel='Nombre moyen de cellules actives',
       ylabel='Erreur maximale sur u.x (m/s)', title='Coût et erreur à t = 0,15 s')
ax.grid(alpha=0.25, which='both')
if ax.collections:
    ax.legend()
else:
    ax.text(0.5, 0.5, 'Erreurs indisponibles : calculer la référence et des configurations.',
            transform=ax.transAxes, ha='center', wrap=True)
plt.show()

## Hauteur moyenne

On compare quelques configurations représentatives et la référence. L'écart maximal en hauteur est calculé sur l'union des instants des deux séries, avec interpolation linéaire dans le temps, sans extrapolation.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), layout='constrained')
if reference['status'] == 'complete':
    h = reference['height']
    ax.plot(h[:, 0], 1e3*h[:, 1], color='black', linewidth=2, label='Référence provisoire')
choices = [(6, '1e-2'), (8, '1e-3'), (9, '1e-3'), (12, '1e-4')]
for key in choices:
    run = runs[key]
    if run['status'] == 'complete':
        h = run['height']
        ax.plot(h[:, 0], 1e3*h[:, 1], label=f'L{key[0]}, U_TOL={key[1]} m/s')
ax.set(xlabel='Temps (s)', ylabel='Hauteur moyenne (mm)', xlim=(0, 0.8),
       title='Montée capillaire : hauteur moyenne')
ax.grid(alpha=0.25)
if ax.lines:
    ax.legend()
else:
    ax.text(0.5, 0.5, 'Aucun résultat représentatif disponible.', transform=ax.transAxes, ha='center')
plt.show()

## Tableau récapitulatif

Un tiret correspond à une donnée manquante. Le statut `complete` exige un résultat vérifié ; il ne garantit pas sa convergence. Les erreurs de la référence par rapport à elle-même valent zéro par définition et ne mesurent pas sa précision.

In [ ]:
def number(value, fmt='.3g'):
    return format(value, fmt) if np.isfinite(value) else '—'

rows = []
for run in [*runs.values(), reference]:
    is_reference = run is reference
    error_u = run.get('error_u', 0. if is_reference and run['status'] == 'complete' else np.nan)
    error_h = run.get('error_h', 0. if is_reference and run['status'] == 'complete' else np.nan)
    status = run['status']
    if status == 'complete' and reference['status'] != 'complete':
        status += ' ; référence absente/invalide'
    if is_reference:
        status = 'référence : ' + status
    rows.append([str(run['level']), number(run['tolerance']), number(error_u), number(error_h),
                 number(run.get('mean_cells', np.nan), '.0f'), number(run.get('wall', np.nan)), status])
headings = ['MAXLEVEL', 'U_TOL (m/s)', 'E∞ sur u.x (m/s)', 'Écart max h (m)',
            'Cellules moyennes', 'Temps réel (s)', 'Statut']
from html import escape
html = '<table><tr>' + ''.join(f'<th>{escape(s)}</th>' for s in headings) + '</tr>'
html += ''.join('<tr>' + ''.join(f'<td>{escape(s)}</td>' for s in row) + '</tr>' for row in rows)
display(HTML(html + '</table>'))